# Multi-Hop Retrieval - Train QueryEncoder (Model 2, retrieval_v3)

**Before running:**
1. Settings -> Accelerator -> **T4 GPU**
2. Settings -> Internet -> **On**
3. Google Drive `musique_data/` folder must contain:
   - `musique_ans_v1.0_train.jsonl`, `musique_ans_v1.0_dev.jsonl`
   - `generator_best.pt`  (from train_generator_kaggle.ipynb, REQUIRED)
4. **(For auto-upload)** Add-ons -> Secrets -> add secret `RCLONE_CONF` (see cell 8)

**What this trains:** QueryEncoder that maps queries into the complement space, so
`dot(M2(Q), G(A,B))` is meaningful. Generator frozen; MarginRankingLoss on
(Q, G(A,B_pos), G(A,B_neg)). Required for a fair FULL eval.

**Expected time: ~1.5 hours on T4**

In [ ]:
# [EDIT IF NEEDED]
REPO_URL        = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME       = "multihop-retrieval"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"   # musique_data folder ID
WORK_DIR        = f"/kaggle/working/{REPO_NAME}/retrieval_v3"
UPLOAD_TO_DRIVE = True    # set False to skip auto-upload

In [ ]:
# 1. Verify GPU - must be T4
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> T4 GPU")
if torch.cuda.get_device_properties(0).major < 7:
    raise RuntimeError("GPU is P100 (sm_60) - change to T4 GPU")
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("CUDA OK")

In [ ]:
# 2. Clone repo + install dependencies
import os
repo_root = f"/kaggle/working/{REPO_NAME}"
if not os.path.isdir(repo_root):
    os.system(f"git clone {REPO_URL} {repo_root}")
else:
    os.system(f"cd {repo_root} && git pull")
os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())
os.system("pip install -q 'transformers>=4.35.0' 'rank_bm25>=0.2.2' gdown")
print("Dependencies ready")

In [ ]:
# 3. Download data + generator checkpoint from Google Drive
import os, gdown
download_dir = "/kaggle/working/drive_data"
if not os.path.isdir(download_dir):
    print("Downloading from Google Drive...")
    gdown.download_folder(id=DRIVE_FOLDER_ID, output=download_dir, quiet=False, use_cookies=False)
else:
    print("Drive data already downloaded")
print("\nDownloaded files:")
for f in sorted(os.listdir(download_dir)):
    print(f"  {f:45s}  {os.path.getsize(f'{download_dir}/{f}')/1e6:7.1f} MB")
m1_path = f"{download_dir}/generator_best.pt"
if not os.path.exists(m1_path):
    raise FileNotFoundError(
        "generator_best.pt not found in Drive. Run train_generator_kaggle.ipynb first "
        "and upload generator_best.pt to musique_data/"
    )
print(f"\ngenerator_best.pt: {os.path.getsize(m1_path)/1e6:.0f} MB  OK")

In [ ]:
# 4. Set up file paths - symlink data into retrieval_v2/, copy generator_best.pt
import os, shutil
drive   = "/kaggle/working/drive_data"
v2_data = f"/kaggle/working/{REPO_NAME}/retrieval_v2/data/musique"
os.makedirs(v2_data,              exist_ok=True)
os.makedirs(f"{WORK_DIR}/models", exist_ok=True)
os.makedirs(f"{WORK_DIR}/cache",  exist_ok=True)
for fname in ["musique_ans_v1.0_train.jsonl", "musique_ans_v1.0_dev.jsonl"]:
    src, dst = f"{drive}/{fname}", f"{v2_data}/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"  {fname}: OK")
m1_dst = f"{WORK_DIR}/models/generator_best.pt"
if not os.path.exists(m1_dst):
    print("  Copying generator_best.pt...", flush=True)
    shutil.copy(f"{drive}/generator_best.pt", m1_dst)
print(f"  generator_best.pt: OK ({os.path.getsize(m1_dst)/1e6:.0f} MB)")
print("\nAll paths ready")

In [ ]:
# 5. Smoke test
import os
os.chdir(WORK_DIR)
print("=== Smoke test (50 examples) ===")
os.system("python model2_train.py --smoke")

---
## Train QueryEncoder - full 3-epoch run

Per step: extract frozen complements `G.extract_complement(A, B_pos/neg)`, encode Q,
MarginRankingLoss: `dot(q, comp_pos) > dot(q, comp_neg) + margin`.

In [ ]:
# 6. Train QueryEncoder (full run)
import os
os.chdir(WORK_DIR)
log_file = "/kaggle/working/model2_train.log"
os.system(f"python model2_train.py 2>&1 | tee {log_file}")
print("\nTraining complete")

In [ ]:
# 7. Verify model2_best.pt + copy to /kaggle/working/ for manual download
import os, shutil
best = f"{WORK_DIR}/models/model2_best.pt"
if os.path.exists(best):
    shutil.copy(best, "/kaggle/working/model2_best.pt")
    print(f"  model2_best.pt: {os.path.getsize(best)/1e6:.1f} MB  OK")
    print("  Copied to /kaggle/working/model2_best.pt")
else:
    print("  model2_best.pt NOT FOUND - check training log")

---
## 8. Auto-upload to Google Drive (rclone)

Uses the same `RCLONE_CONF` secret as the generator notebook. See that notebook's
cell 8 for the one-time local rclone setup. The token stays in Secrets, never in a cell.

In [ ]:
# 8. Upload model2_best.pt straight to Drive musique_data/ folder
import os
if UPLOAD_TO_DRIVE:
    try:
        from kaggle_secrets import UserSecretsClient
        conf = UserSecretsClient().get_secret("RCLONE_CONF")
        os.makedirs("/root/.config/rclone", exist_ok=True)
        with open("/root/.config/rclone/rclone.conf", "w") as fh:
            fh.write(conf)
        os.system("curl -s https://downloads.rclone.org/rclone-current-linux-amd64.zip -o /tmp/r.zip")
        os.system("cd /tmp && unzip -qo r.zip && cp rclone-*-linux-amd64/rclone /usr/bin/ && chmod +x /usr/bin/rclone")
        rc = os.system(
            "rclone copy /kaggle/working/multihop-retrieval/retrieval_v3/models/model2_best.pt "
            f"gdrive: --drive-root-folder-id {DRIVE_FOLDER_ID} -P"
        )
        print("Uploaded to Drive musique_data/" if rc == 0 else f"rclone exited {rc}")
    except Exception as e:
        print("Auto-upload skipped/failed:", e)
        print("Add the RCLONE_CONF secret, or download model2_best.pt from the Output tab.")
else:
    print("UPLOAD_TO_DRIVE=False - download model2_best.pt manually from Output tab.")

---
## Done

If auto-upload ran, `model2_best.pt` is in your Drive `musique_data/` folder.
Next: run `eval_kaggle.ipynb` - FULL now uses Model 2 query encoding (fair test).